In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

df = pd.read_csv("air_fryers_clean_brand_year.csv")

df["log_share"] = np.log(df["brand_share"])

df.head()

,category,year,brand,purchase_count,product_count,avg_price,avg_rating,compact_share,dual_basket_share,oven_style_share,rotisserie_share,window_share,market_purchases,brand_share,log_brand_share,log_share
0,air_fryers,2019,chefman,1146,10,72.963695,4.434119,1.000000,0.0,0.780977,0.243455,0.184119,15076,0.076015,-2.576826,-2.576826
1,air_fryers,2019,cosori,11,2,159.990000,4.581818,1.000000,0.0,0.090909,0.090909,0.000000,15076,0.000730,-7.222964,-7.222964
2,air_fryers,2019,cuisinart,1616,22,229.465274,4.481312,0.993812,0.0,0.889851,0.000000,0.000000,15076,0.107190,-2.233150,-2.233150
3,air_fryers,2019,dash,3011,19,55.176333,4.390767,1.000000,0.0,0.973431,0.000000,0.000000,15076,0.199721,-1.610832,-1.610832
4,air_fryers,2019,gowise usa,4405,45,83.575551,4.552259,0.999773,0.0,0.129398,0.128490,0.000000,15076,0.292186,-1.230364,-1.230364


In [7]:
feature_cols = ['compact_share', 'dual_basket_share', 'oven_style_share', 'rotisserie_share', 'window_share']
print(feature_cols)

['compact_share', 'dual_basket_share', 'oven_style_share', 'rotisserie_share', 'window_share']


In [8]:
brand_dummies = pd.get_dummies(df["brand"], drop_first=True)
year_dummies = pd.get_dummies(df["year"], drop_first=True)

brand_dummies.head()
year_dummies.head()

,2020,2021,2022,2023
0,False,False,False,False
1,False,False,False,False
2,False,False,False,False
3,False,False,False,False
4,False,False,False,False


In [9]:
X = pd.concat([
    df[["avg_price", "avg_rating"]],
    df[feature_cols],
    brand_dummies,
    year_dummies
], axis=1)

X.columns = X.columns.astype(str)

y = df["log_share"]

X.head()

,avg_price,avg_rating,compact_share,dual_basket_share,oven_style_share,rotisserie_share,window_share,cosori,cuisinart,dash,gowise usa,instant_pot,ninja,nuwave,oster,ultrean,2020,2021,2022,2023
0,72.963695,4.434119,1.000000,0.0,0.780977,0.243455,0.184119,False,False,False,False,False,False,False,False,False,False,False,False,False
1,159.990000,4.581818,1.000000,0.0,0.090909,0.090909,0.000000,True,False,False,False,False,False,False,False,False,False,False,False,False
2,229.465274,4.481312,0.993812,0.0,0.889851,0.000000,0.000000,False,True,False,False,False,False,False,False,False,False,False,False,False
3,55.176333,4.390767,1.000000,0.0,0.973431,0.000000,0.000000,False,False,True,False,False,False,False,False,False,False,False,False,False
4,83.575551,4.552259,0.999773,0.0,0.129398,0.128490,0.000000,False,False,False,True,False,False,False,False,False,False,False,False,False


In [10]:
model = LinearRegression()
model.fit(X, y)

results = pd.DataFrame({
    "variable": ["intercept"] + list(X.columns),
    "coefficient": [model.intercept_] + list(model.coef_)
})

results

,variable,coefficient
0,intercept,-13.304892
1,avg_price,-0.037668
2,avg_rating,0.287517
3,compact_share,9.815304
4,dual_basket_share,-9.509686
5,oven_style_share,1.941774
6,rotisserie_share,-5.674054
7,window_share,12.880298
8,cosori,2.551946
9,cuisinart,6.422436


# **Question 1: What is the estimated price coefficient, $\hat{\beta}_{price}$?**


In [13]:
price_coef = results.loc[results['variable'] == 'avg_price', 'coefficient'].iloc[0]
print(f'Price coefficient: {price_coef:.4f}')

Price coefficient: -0.0377


**Question 1 Answer:** The estimated price coefficient is −0.0377. In other words, for every one dollar increase in average price, holding all other factors constant, log market share decreases by 0.0377 log points.

# **Question 2: Is it negative? Why is that important?**

In [14]:
print(f'Price coefficient is negative: {price_coef < 0}')

Price coefficient is negative: True


**Question 2 Answer:** Yes, the price coefficient is negative. This is important because it matches normal demand behavior: when price increases, quantity demanded (and therefore market share) decreases, holding all other factors constant. If the price coefficient was positive, it would imply consumers prefer more expensive products unconditionally, which does not practically apply. A positive price coefficient would also flip the sign of the demand slope for the pricing analysis, which is not possible in the real world.

# **Question 3: Which product features are associated with higher demand?**

In [16]:
feature_results = results[results['variable'].isin(feature_cols)].sort_values('coefficient', ascending=False)
print(feature_results.to_string(index=False))

         variable  coefficient
     window_share    12.880298
    compact_share     9.815304
 oven_style_share     1.941774
 rotisserie_share    -5.674054
dual_basket_share    -9.509686


**Question 3 Answer:** The product features associated with higher demand are window_share (+12.88), followed by compact_share (+9.82), oven_style_share (+1.94), meaning that brands with more window and compact basket products attract higher market shares, holding other factors constant. In contrast, the rotisserie_share (-5.67) and dual_basket_share (-9.51) have negative coefficients, indicating that these more niche features are associated with lower market share. This may reflect the fact that these features are offered by smaller brands with less overall appeal and their effect is already captured by the brand fixed effects. 

# **Question 4: Which brand dummy coefficients are largest? Remember that these are interpreted relative to the dropped brand.**


In [17]:
brand_cols = [col for col in results['variable'] if col in df['brand'].unique().tolist()]

brand_results = results[results['variable'].isin(brand_cols)].sort_values('coefficient', ascending=False)
print(brand_results.to_string(index=False))

   variable  coefficient
  cuisinart     6.422436
      ninja     5.838705
instant_pot     4.626260
 gowise usa     3.938996
      oster     3.928074
     nuwave     3.544883
     cosori     2.551946
    ultrean     0.942399
       dash     0.176655


**Question 4 Answer:** 
The largest brand dummy coefficients are Cuisinart (+6.42), Ninja (+5.84), Instant Pot (+4.63), and Go Wise USA (+3.94), and Oster (+3.93). This indicates that after controlling for price, ratings, and product characteristics, these brands attract significantly higher market shares than Chefman due to brand recognition. 

Dash (+0.18) and Ultrean (+0.94) have the smallest brand coefficients, indicating their demand is barely higher than Chefman's once price and features are accounted for. This suggests these brands compete on price rather than brand strength.

# **Question 5: Which year dummy coefficients are largest? Remember that these are interpreted relative to the dropped year.**

In [18]:
year_cols = ['2020', '2021', '2022', '2023']

year_results = results[results['variable'].isin(year_cols)].sort_values('coefficient', ascending=False)
print(year_results.to_string(index=False))

variable  coefficient
    2020     0.119071
    2021     0.041900
    2023    -0.003307
    2022    -0.098860


**Question 5 Answer:** The year dummy coefficients are interpreted relative to the dropped reference year, 2019. The largest, most positive coefficient was from 2020 of 0.119, suggesting demand was significantly higher in 2020 than 2019 which may have been driven by the COVID pandemic driving consumers to purchase more appliances to cook at home. 2021 also has a slightly positive coefficient of 0.042. 2022 (-0.099) and 2023 (-0.003) have slighltly negative coefficeints, suggesting demand normalized back to 2019 levels after the pandemic period. However, the year effects are relatively small in magnitude which indicates the market structure was relatively stable across years once brand and price effects are controlled for.

# **Question 6: What is the model's $R^2$?**


In [12]:
r2 = model.score(X, y)
print("R^2:", r2)

R^2: 0.7634539500914357


**Question 6 Answer:** The model's $R^2$ is 0.7635, meaning that the model explains 76.35% of the variation in log brand market share. Given that the dataset is comprised of only 50 observations and the model is capturing complex brand and year dynamics through fixed effects, the remaining 24% of variation is attributable to factors not included in the model, which could for example be marketing spend or macroeconomic changes.